In [7]:
class VideoCaptionDataset(Dataset):
    def __init__(self, csv_file, npz_dir, tokenizer):
        self.df = pd.read_csv(csv_file)
        self.npz_dir = npz_dir
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_id = row["video_id"]
        caption = row["caption"]

        npz_path = os.path.join(self.npz_dir, f"{video_id}.npz")
        data = np.load(npz_path)
        
        # Expected shape from npz: (8, 3, 224, 224)
        frames = data["video"].astype(np.float32) / 255.0
        
        # Convert to tensor (Frames, C, H, W)
        frames = torch.tensor(frames)

        tokens = self.tokenizer(
            caption,
            padding="max_length",
            max_length=20,
            truncation=True,
            return_tensors="pt"
        )

        labels = tokens.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": frames, # (8, 3, 224, 224)
            "labels": labels
        }

In [16]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Configuration
INPUT_DIR = "/Users/ayraj/Desktop/video_captioning/processed_frames"
OUTPUT_DIR = "/Users/ayraj/Desktop/video_captioning/final_frames_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Refactoring frames for VideoMAE compatibility...")

for file_name in tqdm(os.listdir(INPUT_DIR)):
    if file_name.endswith(".npz"):
        file_path = os.path.join(INPUT_DIR, file_name)
        
        # 1. Load existing data
        data = np.load(file_path)
        video = data['video'] # Expected: (8, 3, H, W) or (8, H, W, 3)
        
        # 2. Convert to Tensor for easy manipulation
        video_tensor = torch.from_numpy(video).float()
        
        # 3. Ensure Channels are at the correct index (C, T, H, W)
        # If your data is (8, 224, 224, 3), permute(3, 0, 1, 2)
        # If your data is (8, 3, 224, 224), permute(1, 0, 2, 3)
        if video_tensor.shape[1] == 3:
            video_tensor = video_tensor.permute(1, 0, 2, 3)
        elif video_tensor.shape[-1] == 3:
            video_tensor = video_tensor.permute(3, 0, 1, 2)
            
        # 4. Enforce 224x224 resolution
        if video_tensor.shape[-2:] != (224, 224):
            # Interpolate spatial dimensions only
            video_tensor = F.interpolate(video_tensor, size=(224, 224), mode='bilinear')
            
        # 5. Save back to NPZ
        # Shape is now strictly (3, 8, 224, 224)
        output_path = os.path.join(OUTPUT_DIR, file_name)
        np.savez_compressed(output_path, video=video_tensor.numpy().astype(np.uint8))

print(f"Refactoring complete. New files saved in: {OUTPUT_DIR}")

Refactoring frames for VideoMAE compatibility...


100%|██████████| 2000/2000 [01:02<00:00, 32.19it/s]

Refactoring complete. New files saved in: /Users/ayraj/Desktop/video_captioning/final_frames_v2


In [17]:
import numpy as np
import os

# Path to your new frames
CHECK_DIR = "/Users/ayraj/Desktop/video_captioning/final_frames_v2"
sample_file = os.path.join(CHECK_DIR, os.listdir(CHECK_DIR)[0])

data = np.load(sample_file)
frames = data['video']

print(f"File: {sample_file}")
print(f"Shape: {frames.shape}")  # MUST BE (3, 8, 224, 224)
print(f"Data Type: {frames.dtype}")
print(f"Value Range: {frames.min()} to {frames.max()}") # Should be 0-255

if frames.shape == (3, 8, 224, 224):
    print("✅ SUCCESS: Format is usable for VideoMAE.")
else:
    print("❌ ERROR: Shape mismatch. Model will throw a ValueError.")

File: /Users/ayraj/Desktop/video_captioning/final_frames_v2/video2963.npz
Shape: (3, 8, 224, 224)
Data Type: uint8
Value Range: 0 to 255
✅ SUCCESS: Format is usable for VideoMAE.


In [20]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import (
    VideoMAEModel, 
    GPT2Tokenizer, 
    GPT2LMHeadModel, 
    VisionEncoderDecoderModel, 
    GPT2Config,
    VideoMAEImageProcessor # Added for robust preprocessing
)
from peft import LoraConfig, get_peft_model, TaskType

# 1. Setup
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
CSV_PATH = "/Users/ayraj/Desktop/video_captioning/msrvtt_2k_preprocessed.csv"
NPZ_DIR = "/Users/ayraj/Desktop/video_captioning/final_frames_v2"
OUTPUT_DIR = "videomae_gpt2_final_v7"

# Initialize processor to handle shape/channel logic
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")

# 2. Dataset
class VideoMAE_Dataset_Final(Dataset):
    def __init__(self, csv_file, npz_dir, tokenizer):
        self.df = pd.read_csv(csv_file)
        self.npz_dir = npz_dir
        self.tokenizer = tokenizer

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        npz_path = os.path.join(self.npz_dir, f"{row['video_id']}.npz")
        
        with np.load(npz_path) as data:
            video = data['video'] # Current Shape: (3, 8, 224, 224)
            
        # ==========================================
        # THE FIX: Duplicate 8 frames to 16 frames
        # ==========================================
        # np.repeat duplicates each frame adjacently (e.g., AABBCCDDEEFFGGHH)
        video = np.repeat(video, 2, axis=1) # New Shape: (3, 16, 224, 224)
        
        # Processor handles the final normalization
        frames_list = [video[:, i, :, :] for i in range(video.shape[1])]
        pixel_values = processor(frames_list, return_tensors="pt").pixel_values.squeeze(0)
        
        tokens = self.tokenizer(row['caption'], padding='max_length', 
                               max_length=30, truncation=True, return_tensors="pt")
        labels = tokens.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {"pixel_values": pixel_values, "labels": labels}

# 3. Model Initialization
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

decoder_config = GPT2Config.from_pretrained("gpt2")
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True

encoder = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
decoder = GPT2LMHeadModel.from_pretrained("gpt2", config=decoder_config)
model = VisionEncoderDecoderModel(encoder=encoder, decoder=decoder)
#freezing
for param in model.encoder.parameters():
    param.requires_grad = False
print("VideoMAE encoder frozen. Training decoder only.")

# 4. LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, target_modules=["c_attn"], 
    lora_dropout=0.1
)
model.decoder = get_peft_model(model.decoder, lora_config)

model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.to(DEVICE)

# 5. Training Loop
train_loader = DataLoader(VideoMAE_Dataset_Final(CSV_PATH, NPZ_DIR, tokenizer), 
                          batch_size=2, shuffle=True) # Reduced batch size for stability
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(5):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Use the processor-verified pixel_values directly
        outputs = model(pixel_values=pixel_values, labels=labels)
        
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

# 6. Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Loading weights: 100%|██████████| 184/184 [00:00<00:00, 16430.04it/s]
VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.head.weight                                                  | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED | 

KeyboardInterrupt: 